# 19_recommendations: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/19_recommendations/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compute MMR from relevance minus similarity to already selected items.

Run: uv run  python modules/19_recommendations/examples/01_from_scratch.py
Expected: high diversity selects a different second item.
CPU: seconds to one minute. Needs: base, NumPy. No implicit downloads.
"""

import numpy as np

from common.apps.recommendations import mmr


def main() -> None:
    ids = ["red-mug", "red-mug-copy", "blue-travel-cup"]
    vectors = np.array([[1.0, 0.0], [0.99, 0.01], [0.0, 1.0]])
    # KEY IDEA: selecting an item changes the redundancy penalty on the next iteration.
    for relevance in [1.0, 0.7, 0.2]:
        print(relevance, mmr(ids, vectors, [1, 0], k=2, relevance=relevance))
    assert mmr(ids, vectors, [1, 0], 2, 0.2)[1] == "blue-travel-cup"


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/19_recommendations/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None:
    print("SKIP native comparison: --extra qdrant")
else:
    """Query a real database with an event-weighted user vector.

    Run: uv run --extra qdrant python modules/19_recommendations/examples/02_production.py
    Expected: recommendations exclude products already seen before the cutoff.
    CPU: seconds to one minute. Needs: qdrant extra, saved catalog and events. No implicit downloads.
    """
    # course-extra: qdrant

    import pandas as pd

    from common.apps.database import AppDatabase
    from common.apps.product_search import ProductSearch
    from common.apps.recommendations import recommend, user_profile
    from common.settings import data_dir

    def main() -> None:
        events = pd.read_parquet(data_dir() / "tier4/events.parquet")
        with AppDatabase("qdrant-local") as db:
            search = ProductSearch(db, 1000)
            vectors = {key: r.vector for key, r in search.by_id.items()}
            group = events[events.user_id == "synthetic-user-0000"]
            profile, seen = user_profile(group.to_dict("records"), vectors, "2025-12-20T00:00:00Z")
            assert profile is not None
            # KEY IDEA: future purchases cannot leak into the user profile.
            ids = recommend(search, profile, exclude=seen)
            assert not set(ids) & seen
            print({"seen": sorted(seen), "recommendations": ids})

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/19_recommendations/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Choose a duplicate threshold using development families only.

Run: uv run  python modules/19_recommendations/examples/03_measure.py
Expected: development threshold and held-out precision/recall.
CPU: seconds to one minute. Needs: base, duplicate pairs and catalog vectors. No implicit downloads.
"""

import pandas as pd

from common.apps.deduplication import calibrate, pair_scores, threshold_metrics
from common.settings import data_dir
from common.teaching import catalog, evidence


def main() -> None:
    products, matrix, _, _ = catalog(10000)
    vectors = dict(zip(products.id, matrix, strict=True))
    pairs = pd.read_parquet(data_dir() / "tier4/duplicates.parquet")
    pairs = pairs[pairs.source_id.isin(vectors) & pairs.duplicate_id.isin(vectors)]
    dev, test = pairs[pairs.split == "development"], pairs[pairs.split == "test"]
    assert not set(dev.group_id) & set(test.group_id)
    chosen = calibrate(pair_scores(dev.to_dict("records"), vectors), dev.label.tolist())
    # KEY IDEA: freeze the threshold before looking at held-out outcomes.
    measured = threshold_metrics(
        pair_scores(test.to_dict("records"), vectors), test.label.tolist(), chosen["threshold"]
    )
    print({"development": chosen, "test": measured})
    evidence("duplicates", {"development": chosen, "test": measured}, tier=4)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 04_review_themes

Source: `modules/19_recommendations/examples/04_review_themes.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Cluster real review embeddings and inspect high-distance outliers.

Run: uv run  python modules/19_recommendations/examples/04_review_themes.py
Expected: six themes, representative text and ten candidate outliers.
CPU: seconds to one minute. Needs: base, prepared review vectors. No implicit downloads.
"""

from common.apps.clustering import review_themes
from common.teaching import evidence
from common.tier4 import review_data


def main() -> None:
    reviews, vectors, _ = review_data()
    result = review_themes(reviews.text.tolist(), vectors)
    for theme in result["themes"]:
        print(theme)
    # KEY IDEA: distance is a review priority, not a fraud or correctness label.
    print(reviews.iloc[result["outliers"]][["id", "text"]].to_dict("records"))
    evidence("review-themes", result, tier=4)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.apps.recommendations import mmr

print(mmr(["a", "b", "c"], [[1, 0], [0.99, 0.01], [0, 1]], [1, 0], 2, 0.2))
# Change relevance to 1 and explain the second choice.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.